# Extract Mispronunciations
To prepare for evaluation, specifically extracting False positive and False Negative rates, we must first establish ground truth labels for our L2-speaker data compared to a canonical source. To do this, we first align the phonemes of the 

# Imports

In [ ]:
import jiwer

_IDENTITY = jiwer.transforms.Compose([])


# Define Operations


In [ ]:
def align_phones(canonical: list[str], annotated: list[str]) -> list[dict]:
    """
    Align annotated phones to canonical using jiwer.
    Returns one dict per alignment event with keys: canonical, annotated, op.
    """
    result = jiwer.process_words(
        reference=" ".join(canonical),
        hypothesis=" ".join(annotated),
        reference_transform=_IDENTITY,
        hypothesis_transform=_IDENTITY,
    )

    aligned = []
    for chunk in result.alignments[0]:
        can_chunk = canonical[chunk.ref_start_idx:chunk.ref_end_idx]
        ann_chunk = annotated[chunk.hyp_start_idx:chunk.hyp_end_idx]

        if chunk.type == 'equal':
            for c, a in zip(can_chunk, ann_chunk):
                aligned.append({'canonical': c, 'annotated': a, 'op': 'match'})

        elif chunk.type == 'substitute':
            for c, a in zip(can_chunk, ann_chunk):
                aligned.append({'canonical': c, 'annotated': a, 'op': 'substitution'})

        elif chunk.type == 'delete':
            for c in can_chunk:
                aligned.append({'canonical': c, 'annotated': None, 'op': 'deletion'})

        elif chunk.type == 'insert':
            for a in ann_chunk:
                aligned.append({'canonical': None, 'annotated': a, 'op': 'insertion'})

    return aligned


def phone_error_rate(canonical: list[str], annotated: list[str]) -> float:
    """PER computed directly from jiwer — no need to pre-align."""
    result = jiwer.process_words(
        reference=" ".join(canonical),
        hypothesis=" ".join(annotated),
        reference_transform=_IDENTITY,
        hypothesis_transform=_IDENTITY,
    )
    return result.wer


# Ground truth label per canonical phone position
# True = mispronounced, False = correct, None = insertion (no canonical slot)
OP_LABEL = {'match': False, 'substitution': True, 'deletion': True, 'insertion': None}
